<h1 style="font-weight:900; color:#0E2841; margin:0.4em 0 0.2em;">Lerneinheit 2 · Vom Text zum Tensor</h1>

<div style="font-size:0.8em; color:#5a6472; margin-top:-4px;">Begleit-Notebook · Version 1.0</div>

<span style="color:#5a6472; font-size:0.9em;">Architektur Neuronaler Netze für Generative KI 2 · Prof. Dr. Stephan Kurpjuweit · Hochschule Worms · © 2026</span>

<div style="border-radius:6px; overflow:hidden; margin:2px 0; margin-right:-20px;">
<div style="background:#E8B029; color:#fff; padding:6px 16px; font-weight:bold; font-size:1.08em; margin:0; overflow:hidden;">Lernziele</div>
<div style="background:#FDF6E3; color:#1a1a1a; padding:12px 16px; margin:0;">
<p>Nach dieser Lerneinheit können Sie …</p>
<ul>
<li>erklären, was ein autoregressives Sprachmodell berechnet und warum es ohne annotierte Daten lernt,</li>
<li>Byte Pair Encoding von Hand ausführen und begründen, warum Byte-Level-BPE kein unbekanntes Token braucht,</li>
<li>abschätzen, wie Sprache und Tokenizer Länge, Kosten und Kontext bestimmen,</li>
<li>aus einem Text mit dem Sliding Window Paare aus Eingabe und Ziel für das Training bilden,</li>
<li>die Formen <code>[B, T]</code> und <code>[B, T, d]</code> sowie die Parameterzahl der Embedding-Tabellen herleiten.</li>
</ul>

</div>
</div>

Die Folien erklären, was ein Sprachmodell sieht, wenn es Text bekommt. In diesem Notebook steht der Code dazu: vom Rohtext über Tokens und Token-IDs bis zum Tensor, den das Modell als Eingabe erhält. Der Code zu den Kapiteln 2.2 bis 2.4 stammt überwiegend aus Kapitel 2 von Sebastian Raschkas Buch *Build a Large Language Model (From Scratch)*.

Jedes Kapitel hat zwei Teile. Im ersten Teil lesen wir fertigen Code gemeinsam in der Vorlesung: Bei „Kurz überlegen" legen Sie sich fest, dann führen Sie die Zelle aus. Der zweite Teil heißt **Übungen**. Die Prüfzellen zeigen, ob Ihre Lösung stimmt. In der Vorlesung führen wir die Zellen einzeln aus und überspringen die Übungen zunächst. Keine Zelle des ersten Teils hängt von einer Übung ab. Was im Termin offen bleibt, bearbeiten Sie bis zum nächsten Termin.

<div style="border-radius:6px; overflow:hidden; margin:2px 0; margin-right:-20px;">
<div style="background:#E8B029; color:#fff; padding:6px 16px; font-weight:bold; font-size:1.08em; margin:0; overflow:hidden;">Inhalt</div>
<div style="background:#FDF6E3; color:#1a1a1a; padding:12px 16px; margin:0;">
<ul style="margin:0; padding-left:1.2em;"><li style="margin:5px 0;">2.1&nbsp;&nbsp;<a href="#kap-sprachmodelle-durch-zaehlen" style="color:#1a1a1a; text-decoration:none; border-bottom:1px solid #E8B029;">Sprachmodelle durch Zählen</a><span class="modulchip" style="font-size:0.55em; font-weight:bold; letter-spacing:0.02em; border:1.5px solid #475569; color:#475569; padding:1px 8px; border-radius:10px; margin-left:12px; vertical-align:0.35em; white-space:nowrap;">Folien F06 und F07</span></li><li style="margin:5px 0;">2.2&nbsp;&nbsp;<a href="#kap-tokenisierung" style="color:#1a1a1a; text-decoration:none; border-bottom:1px solid #E8B029;">Tokenisierung</a><span class="modulchip" style="font-size:0.55em; font-weight:bold; letter-spacing:0.02em; border:1.5px solid #475569; color:#475569; padding:1px 8px; border-radius:10px; margin-left:12px; vertical-align:0.35em; white-space:nowrap;">Raschka 2.2 bis 2.5</span></li><li style="margin:5px 0;">2.3&nbsp;&nbsp;<a href="#kap-trainingspaare-mit-dem-sliding-window" style="color:#1a1a1a; text-decoration:none; border-bottom:1px solid #E8B029;">Trainingspaare mit dem Sliding Window</a><span class="modulchip" style="font-size:0.55em; font-weight:bold; letter-spacing:0.02em; border:1.5px solid #475569; color:#475569; padding:1px 8px; border-radius:10px; margin-left:12px; vertical-align:0.35em; white-space:nowrap;">Raschka 2.6</span></li><li style="margin:5px 0;">2.4&nbsp;&nbsp;<a href="#kap-embeddings" style="color:#1a1a1a; text-decoration:none; border-bottom:1px solid #E8B029;">Embeddings</a><span class="modulchip" style="font-size:0.55em; font-weight:bold; letter-spacing:0.02em; border:1.5px solid #475569; color:#475569; padding:1px 8px; border-radius:10px; margin-left:12px; vertical-align:0.35em; white-space:nowrap;">Raschka 2.7 und 2.8</span></li></ul>

</div>
</div>

<div style="border-radius:6px; overflow:hidden; margin:2px 0; margin-right:-20px;">
<div style="background:#2E75B6; color:#fff; padding:6px 16px; font-weight:bold; font-size:1.08em; margin:0; overflow:hidden;">Bevor es losgeht</div>
<div style="background:#EAF2FA; color:#1a1a1a; padding:12px 16px; margin:0;">
<p>In <strong>Colab</strong> klicken Sie zuerst auf „Kopie in Drive speichern". Ohne diesen Klick gehen Ihre Änderungen beim Schließen verloren.</p>
<p><strong>Lokal</strong> starten Sie JupyterLab über das Startskript im Kursordner (<code>start_lokal</code>). Die README im Kursordner beschreibt beide Wege.</p>
<p>Die folgende Zelle prüft Ihre Umgebung. In Colab installiert sie dabei das fehlende Paket <code>tiktoken</code>. Eine GPU brauchen Sie für diese Lerneinheit nicht.</p>

</div>
</div>

In [ ]:
# Vorgabe: Umgebung prüfen — einfach ausführen
import importlib
import subprocess
import sys
from importlib.metadata import version

IN_COLAB = "google.colab" in sys.modules
v = sys.version_info
print(f"{'✓' if (v.major, v.minor) >= (3, 10) else '✗'} Python    {v.major}.{v.minor}.{v.micro}")

for paket in ["torch", "tiktoken"]:
    try:
        importlib.import_module(paket)
    except ModuleNotFoundError:
        if not IN_COLAB:
            print(f"✗ {paket:9s} fehlt. Bitte JupyterLab über das Startskript starten (siehe README).")
            continue
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paket], check=True)
    print(f"✓ {paket:9s} {version(paket)}")

if "torch" in sys.modules:
    import torch
    if torch.cuda.is_available():
        geraet = f"CUDA-GPU ({torch.cuda.get_device_name(0)})"
    elif torch.backends.mps.is_available():
        geraet = "Apple-GPU (MPS)"
    else:
        geraet = "CPU"
    print(f"✓ Gerät     {geraet}")

In [ ]:
# Vorgabe: Pakete und Texte laden — einfach ausführen
import os
import random
import re
import urllib.request
from collections import Counter, defaultdict

import tiktoken
import torch

KURS_REPO = "https://raw.githubusercontent.com/skuhswo/ann2/main/LE02/"
RASCHKA = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/"


def lade_text(datei, quelle):
    if not os.path.exists(datei):
        urllib.request.urlretrieve(quelle + datei, datei)
    with open(datei, "r", encoding="utf-8") as f:
        return f.read()


raw_text = lade_text("the-verdict.txt", RASCHKA)
try:
    goethe = lade_text("goethe.txt", KURS_REPO)
except Exception:
    goethe = raw_text
    print("goethe.txt nicht gefunden. Die Beispiele mit deutschem Text laufen mit dem englischen Text.")

print("The Verdict:", len(raw_text), "Zeichen")
print("Goethe:     ", len(goethe), "Zeichen")

Zwei Texte begleiten das Notebook. *The Verdict* von Edith Wharton ist die englische Kurzgeschichte aus Raschkas Buch. `goethe.txt` enthält deutsche Texte von Johann Wolfgang von Goethe. Beide sind gemeinfrei und liegen neben diesem Notebook. Fehlt eine Datei, etwa in Colab, lädt die Zelle sie nach.

## <a id="kap-sprachmodelle-durch-zaehlen"></a><span class="zwischentitel" style="display:block; color:#0E2841; border-bottom:3px solid #156082; padding-bottom:4px;  padding-top:26px;"><span style="float:right"><span class="modulchip" style="font-size:0.55em; font-weight:bold; letter-spacing:0.02em; border:1.5px solid #475569; color:#475569; padding:1px 8px; border-radius:10px; margin-left:12px; vertical-align:0.35em; white-space:nowrap;">Folien F06 und F07</span></span>2.1&nbsp;&nbsp;Sprachmodelle durch Zählen</span>

Ein autoregressives Sprachmodell gibt an, wie wahrscheinlich jedes mögliche nächste Zeichen oder Token ist. Die einfachste Bauart kommt ohne neuronales Netz aus: Sie zählt, welches Zeichen im Text wie oft auf einen Kontext aus `n` Zeichen folgt. Zum Erzeugen zieht sie das nächste Zeichen entsprechend dieser Häufigkeiten, hängt es an und wiederholt den Schritt.

In [ ]:
# Vorgabe: N-Gramm-Sprachmodell — einfach ausführen
def ngramm_modell(text, n):
    """Zählt für jeden Kontext aus n Zeichen, welches Zeichen wie oft folgt."""
    tabelle = defaultdict(Counter)
    for i in range(len(text) - n):
        tabelle[text[i:i + n]][text[i + n]] += 1
    return tabelle


def erzeuge(tabelle, n, start="Die Sonne ", laenge=120, seed=7):
    """Zieht Zeichen für Zeichen aus der gezählten Verteilung."""
    random.seed(seed)
    ausgabe = start
    for _ in range(laenge):
        folge = tabelle.get(ausgabe[-n:] if n else "")
        if not folge:               # ungesehener Kontext: keine Fortsetzung
            break
        ausgabe += random.choices(list(folge), weights=list(folge.values()))[0]
    return ausgabe


tabelle = ngramm_modell(goethe, 4)
for kontext in ("eine", "Mens"):
    folge = tabelle[kontext]
    gesamt = sum(folge.values())
    print(repr(kontext), "->", [(zeichen, round(anzahl / gesamt, 2)) for zeichen, anzahl in folge.most_common(5)])

Die Ausgabe ist die Verteilung über das nächste Zeichen für zwei Kontexte. Nach `eine` kommen mehrere Zeichen in Frage, nach `Mens` praktisch nur eines.

<div style="border:2px solid #E97132; background:#fff; color:#1a1a1a; padding:12px 16px; border-radius:6px; margin-right:-20px;">
<div style="overflow:hidden; margin-bottom:4px;"><span style="color:#E97132; font-weight:bold; font-size:1.08em;">💭 Kurz überlegen</span></div>
<p>Das Modell erzeugt Text mit 0, 2, 4 und 8 Zeichen Kontext. Ab wie vielen Zeichen Kontext entstehen deutsche Wörter?</p>

</div>

In [ ]:
# Vorgabe: Text erzeugen — einfach ausführen
for n in (0, 2, 4, 8):
    tabelle = ngramm_modell(goethe, n)
    print(f"n = {n}, Kontexte in der Tabelle: {len(tabelle)}")
    print("   ", erzeuge(tabelle, n).replace("\n", " "))

Mit wachsendem Kontext entstehen erst Wörter, dann Satzteile. Zugleich wächst die Tabelle stark an, und für einen Kontext, der im Text nie vorkam, gibt es keine Fortsetzung. Ein neuronales Sprachmodell ersetzt die Tabelle durch gelernte Vektoren und kann deshalb auch auf ähnliche, nie gesehene Kontexte antworten.

### <span class="zwischentitel" style="display:block; color:#0E2841; border-bottom:1px solid rgba(21,96,130,0.45); padding-bottom:2px;  padding-top:16px;">2.1.1&nbsp;&nbsp;Übungen</span>

<div style="border-radius:6px; overflow:hidden; margin:2px 0; margin-right:-20px;">
<div style="background:#2E7D32; color:#fff; padding:6px 16px; font-weight:bold; font-size:1.08em; margin:0; overflow:hidden;"><span style="float:right; font-weight:600; font-size:0.85em; opacity:0.95;">5 min</span>✏️ Aufgabe 2.1 #1</div>
<div style="background:#EAF4EB; color:#1a1a1a; padding:12px 16px; margin:0;">
<p>Wie viele verschiedene Kontexte kennt die Tabelle bei <code>n = 3</code>? Und wie viele Kontexte wären möglich, wenn jedes Zeichen des Textes an jeder der drei Stellen stehen könnte? Speichern Sie die Werte in <code>kontexte_3</code> und <code>moeglich_3</code>.</p>
<details style="margin-top:10px"><summary style="cursor:pointer; color:#2E7D32; font-weight:600; font-size:0.9em">Hinweis anzeigen</summary><div style="margin-top:8px; padding-top:8px; border-top:1px dashed #2E7D3255"><p><strong>Hinweis:</strong> <code>len(set(goethe))</code> liefert die Anzahl der verschiedenen Zeichen im Text.</p></div></details>

</div>
</div>

In [ ]:
# TODO: Ihre Lösung hier
kontexte_3 = ...
moeglich_3 = ...

In [ ]:
# Prüfzelle — einfach ausführen
assert kontexte_3 == len(ngramm_modell(goethe, 3)), "Zählen Sie die Einträge der Tabelle für n = 3"
assert moeglich_3 == len(set(goethe)) ** 3, "Anzahl der verschiedenen Zeichen hoch 3"
print("Alle Prüfungen bestanden ✓")

<div style="border:2px solid #7E3FA0; background:#fff; color:#1a1a1a; padding:12px 16px; border-radius:6px; ">
<div style="overflow:hidden; margin-bottom:4px;"><span style="color:#7E3FA0; font-weight:bold; font-size:1.08em;">Lernfragen 2.1 #1</span></div>
<p style="padding-left:1.2em; text-indent:-1.2em;"><b style="color:#7E3FA0;">1.</b> Nur ein kleiner Teil der möglichen Kontexte kommt im Text vor. Was bedeutet das für ein Zählmodell mit langem Kontext?</p>
<p style="padding-left:1.2em; text-indent:-1.2em;"><b style="color:#7E3FA0;">2.</b> Woher kommen bei diesem Modell die Trainingsdaten und die Labels?</p>

</div>

## <a id="kap-tokenisierung"></a><span class="zwischentitel" style="display:block; color:#0E2841; border-bottom:3px solid #156082; padding-bottom:4px;  padding-top:26px;"><span style="float:right"><span class="modulchip" style="font-size:0.55em; font-weight:bold; letter-spacing:0.02em; border:1.5px solid #475569; color:#475569; padding:1px 8px; border-radius:10px; margin-left:12px; vertical-align:0.35em; white-space:nowrap;">Raschka 2.2 bis 2.5</span></span>2.2&nbsp;&nbsp;Tokenisierung</span>

### <span class="zwischentitel" style="display:block; color:#0E2841; border-bottom:1px solid rgba(21,96,130,0.45); padding-bottom:2px;  padding-top:16px;">2.2.1&nbsp;&nbsp;Ein Tokenizer von Hand</span>

Der einfachste Tokenizer trennt an Leerzeichen und Satzzeichen. Der reguläre Ausdruck behält die Satzzeichen als eigene Tokens, leere Stücke werden verworfen.

In [ ]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])

Das Vokabular ordnet jedem verschiedenen Token eine ganze Zahl zu. Der Tokenizer bekommt zwei Methoden: `encode` wandelt Text in Token-IDs um, `decode` wandelt Token-IDs zurück in Text.

In [ ]:
# Vorgabe: Vokabular und Tokenizer — einfach ausführen
all_words = sorted(set(preprocessed))
vocab = {token: integer for integer, token in enumerate(all_words)}


class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)      # Leerzeichen vor Satzzeichen entfernen
        return text


tokenizer = SimpleTokenizerV1(vocab)
ids = tokenizer.encode("\"It's the last he painted, you know,\" Mrs. Gisburn said with pardonable pride.")
print(ids)
print(tokenizer.decode(ids))

<div style="border:2px solid #E97132; background:#fff; color:#1a1a1a; padding:12px 16px; border-radius:6px; margin-right:-20px;">
<div style="overflow:hidden; margin-bottom:4px;"><span style="color:#E97132; font-weight:bold; font-size:1.08em;">💭 Kurz überlegen</span></div>
<p>Der Satz in der nächsten Zelle stammt nicht aus der Kurzgeschichte. Was passiert beim Kodieren?</p>

</div>

In [ ]:
try:
    print(tokenizer.encode("Hello, do you like tea?"))
except KeyError as fehler:
    print("KeyError:", fehler)

### <span class="zwischentitel" style="display:block; color:#0E2841; border-bottom:1px solid rgba(21,96,130,0.45); padding-bottom:2px;  padding-top:16px;">2.2.2&nbsp;&nbsp;Sondertokens</span>

Zwei zusätzliche Einträge im Vokabular fangen solche Fälle ab. `<|unk|>` steht für jedes unbekannte Wort. `<|endoftext|>` trennt unabhängige Texte, die zu einem langen Trainingstext verkettet werden.

### <span class="zwischentitel" style="display:block; color:#0E2841; border-bottom:1px solid rgba(21,96,130,0.45); padding-bottom:2px;  padding-top:16px;">2.2.3&nbsp;&nbsp;Byte Pair Encoding lernen</span>

Byte Pair Encoding lernt sein Vokabular aus einem Korpus: mit Einzelzeichen beginnen, das häufigste Nachbarpaar zählen, verschmelzen, als neues Token aufnehmen, wiederholen. Das Mini-Korpus stammt von den Folien.

<div style="border:2px solid #E97132; background:#fff; color:#1a1a1a; padding:12px 16px; border-radius:6px; margin-right:-20px;">
<div style="overflow:hidden; margin-bottom:4px;"><span style="color:#E97132; font-weight:bold; font-size:1.08em;">💭 Kurz überlegen</span></div>
<p>Das Korpus enthält <code>lernen</code> (5-mal), <code>lernt</code> (4-mal), <code>gelernt</code> (3-mal), <code>lesen</code> (2-mal) und <code>gelesen</code> (2-mal). Welches Nachbarpaar wird als erstes verschmolzen?</p>

</div>

In [ ]:
# Vorgabe: BPE lernen — einfach ausführen
def verschmelze(symbole, a, b):
    """Ersetzt jedes Nachbarpaar (a, b) durch das neue Symbol a + b."""
    neu, i = [], 0
    while i < len(symbole):
        if i < len(symbole) - 1 and symbole[i] == a and symbole[i + 1] == b:
            neu.append(a + b)
            i += 2
        else:
            neu.append(symbole[i])
            i += 1
    return tuple(neu)


def lerne_bpe(korpus, anzahl_merges):
    """korpus: Wort -> Häufigkeit. Liefert die Liste der Merges und die zerlegten Wörter."""
    woerter = {tuple(wort): anzahl for wort, anzahl in korpus.items()}
    merges = []
    for _ in range(anzahl_merges):
        paare = Counter()
        for symbole, anzahl in woerter.items():
            for a, b in zip(symbole, symbole[1:]):
                paare[(a, b)] += anzahl
        if not paare:
            break
        (a, b), haeufigkeit = max(paare.items(), key=lambda eintrag: eintrag[1])
        merges.append((a, b, haeufigkeit))
        woerter = {verschmelze(symbole, a, b): anzahl for symbole, anzahl in woerter.items()}
    return merges, woerter


def zerlege(wort, merges):
    """Wendet die gelernten Merges in ihrer Reihenfolge auf ein neues Wort an."""
    symbole = tuple(wort)
    for a, b, _ in merges:
        symbole = verschmelze(symbole, a, b)
    return list(symbole)


korpus = {"lernen": 5, "lernt": 4, "gelernt": 3, "lesen": 2, "gelesen": 2}
merges, woerter = lerne_bpe(korpus, 6)
for schritt, (a, b, haeufigkeit) in enumerate(merges, 1):
    print(f"{schritt}. {a} + {b} -> {a + b}   ({haeufigkeit}-mal)")
print("verlernt ->", zerlege("verlernt", merges))

Der Algorithmus entspricht Algorithmus 1 aus Sennrich et al. (2016). Das Wort `verlernt` kam im Korpus nicht vor und lässt sich trotzdem darstellen. Dass dabei der Wortstamm `lern` als Stück entsteht, folgt aus den Häufigkeiten. BPE kennt keine Grammatik.

### <span class="zwischentitel" style="display:block; color:#0E2841; border-bottom:1px solid rgba(21,96,130,0.45); padding-bottom:2px;  padding-top:16px;">2.2.4&nbsp;&nbsp;Byte Pair Encoding mit tiktoken</span>

GPT-2 zerlegt Text mit Byte Pair Encoding (BPE) in Subwords. Wir nutzen dafür `tiktoken`, die Implementierung von OpenAI. Sondertokens müssen beim Kodieren ausdrücklich erlaubt werden.

In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")
print("Vokabulargröße:", tokenizer.n_vocab)

integers = tokenizer.encode("Hello, do you like tea? <|endoftext|> In the sunlit terraces",
                            allowed_special={"<|endoftext|>"})
print(integers)
print(tokenizer.decode(integers))

Die Hilfsfunktion zeigt, in welche Stücke ein Text zerfällt. Das Zeichen `�` bedeutet, dass ein Token nur einen Teil eines Mehrbyte-Zeichens enthält, zum Beispiel ein halbes `ü`.

In [ ]:
# Vorgabe: Hilfsfunktion — einfach ausführen
def zeige_tokens(text):
    ids = tokenizer.encode(text)
    teile = [tokenizer.decode([i]) for i in ids]
    print(f"{len(ids):3d} Tokens: {teile}")
    return ids


_ = zeige_tokens("Hello world")
_ = zeige_tokens(" Hello world")
_ = zeige_tokens("hello world")

<div style="border:2px solid #E97132; background:#fff; color:#1a1a1a; padding:12px 16px; border-radius:6px; margin-right:-20px;">
<div style="overflow:hidden; margin-bottom:4px;"><span style="color:#E97132; font-weight:bold; font-size:1.08em;">💭 Kurz überlegen</span></div>
<p>Der deutsche Satz in der nächsten Zelle hat 44 Zeichen, der englische 53. Welcher braucht mehr Tokens?</p>

</div>

In [ ]:
# Vorgabe: Laufbeispiel — einfach ausführen
satz_de = "Übermorgen fährt das Rheinschiff nach Worms."
satz_en = "The day after tomorrow the Rhine ship sails to Worms."

for satz in (satz_de, satz_en):
    print(f"{len(satz)} Zeichen, {len(satz.encode('utf-8'))} Bytes")
    _ = zeige_tokens(satz)

Das große `Ü` hat im GPT-2-Vokabular kein eigenes Token und zerfällt in seine zwei Bytes. Wie stark sich Deutsch und Englisch im Mittel unterscheiden, zeigt die nächste Zelle an längeren Texten.

In [ ]:
# Vorgabe: Zeichen je Token — einfach ausführen
for name, probe in (("The Verdict (englisch)", raw_text), ("Goethe (deutsch)", goethe[:200_000])):
    anzahl = len(tokenizer.encode(probe))
    print(f"{name}: {len(probe)} Zeichen, {anzahl} Tokens, {len(probe) / anzahl:.2f} Zeichen je Token")

### <span class="zwischentitel" style="display:block; color:#0E2841; border-bottom:1px solid rgba(21,96,130,0.45); padding-bottom:2px;  padding-top:16px;">2.2.5&nbsp;&nbsp;Übungen</span>

<div style="border-radius:6px; overflow:hidden; margin:2px 0; margin-right:-20px;">
<div style="background:#2E7D32; color:#fff; padding:6px 16px; font-weight:bold; font-size:1.08em; margin:0; overflow:hidden;"><span style="float:right; font-weight:600; font-size:0.85em; opacity:0.95;">3 min</span>✏️ Aufgabe 2.2 #1</div>
<div style="background:#EAF4EB; color:#1a1a1a; padding:12px 16px; margin:0;">
<p>Bestimmen Sie die Anzahl der Tokens im Text und die Anzahl der <strong>verschiedenen</strong> Tokens. Speichern Sie die Ergebnisse in <code>anzahl_tokens</code> und <code>anzahl_verschiedene</code>.</p>
<details style="margin-top:10px"><summary style="cursor:pointer; color:#2E7D32; font-weight:600; font-size:0.9em">Hinweis anzeigen</summary><div style="margin-top:8px; padding-top:8px; border-top:1px dashed #2E7D3255"><p><strong>Hinweis:</strong> Ein <code>set</code> enthält jeden Wert nur einmal.</p></div></details>

</div>
</div>

In [ ]:
# TODO: Ihre Lösung hier
anzahl_tokens = ...
anzahl_verschiedene = ...

In [ ]:
# Prüfzelle — einfach ausführen
assert anzahl_tokens == 4690, "Zählen Sie die Einträge der Liste preprocessed"
assert anzahl_verschiedene == 1130, "Jedes Token darf nur einmal gezählt werden"
print("Alle Prüfungen bestanden ✓")

<div style="border-radius:6px; overflow:hidden; margin:2px 0; margin-right:-20px;">
<div style="background:#2E7D32; color:#fff; padding:6px 16px; font-weight:bold; font-size:1.08em; margin:0; overflow:hidden;"><span style="float:right; font-weight:600; font-size:0.85em; opacity:0.95;">5 min</span>✏️ Aufgabe 2.2 #2</div>
<div style="background:#EAF4EB; color:#1a1a1a; padding:12px 16px; margin:0;">
<p>Ergänzen Sie die Methode <code>encode</code> so, dass jedes Token, das nicht im Vokabular steht, durch <code>&lt;|unk|&gt;</code> ersetzt wird. Eine Zeile genügt.</p>
<p>Wer früher fertig ist: Kodieren und dekodieren Sie einen eigenen Satz. Welche Wörter gehen verloren?</p>
<details style="margin-top:10px"><summary style="cursor:pointer; color:#2E7D32; font-weight:600; font-size:0.9em">Hinweis anzeigen</summary><div style="margin-top:8px; padding-top:8px; border-top:1px dashed #2E7D3255"><p><strong>Hinweis:</strong> Eine List Comprehension mit Bedingung prüft für jedes Element <code>item in self.str_to_int</code>.</p></div></details>

</div>
</div>

In [ ]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
vocab = {token: integer for integer, token in enumerate(all_tokens)}


class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]

        # TODO: Ihre Lösung hier
        ...

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

In [ ]:
# Prüfzelle — einfach ausführen
_tok = SimpleTokenizerV2(vocab)
_text = " <|endoftext|> ".join(("Hello, do you like tea?", "In the sunlit terraces of the palace."))
_ids = _tok.encode(_text)
print(_tok.decode(_ids))
assert len(vocab) == 1132, "Das Vokabular enthält jetzt zwei Sondertokens mehr"
assert _ids[0] == vocab["<|unk|>"], "Hello ist unbekannt und bekommt die ID von <|unk|>"
assert _ids == [1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]
print("Alle Prüfungen bestanden ✓")

<div style="border:2px solid #7E3FA0; background:#fff; color:#1a1a1a; padding:12px 16px; border-radius:6px; ">
<div style="overflow:hidden; margin-bottom:4px;"><span style="color:#7E3FA0; font-weight:bold; font-size:1.08em;">Lernfragen 2.2 #1</span></div>
<p style="padding-left:1.2em; text-indent:-1.2em;"><b style="color:#7E3FA0;">1.</b> Vergleichen Sie den dekodierten Text aus der Prüfzelle mit dem Original. Welche Information ist verloren gegangen?</p>
<p style="padding-left:1.2em; text-indent:-1.2em;"><b style="color:#7E3FA0;">2.</b> Warum ist <code>&lt;|unk|&gt;</code> für ein Modell, das Texte <strong>erzeugen</strong> soll, ein größeres Problem als für ein Modell, das Texte nur einordnet?</p>

</div>

<div style="border-radius:6px; overflow:hidden; margin:2px 0; margin-right:-20px;">
<div style="background:#2E7D32; color:#fff; padding:6px 16px; font-weight:bold; font-size:1.08em; margin:0; overflow:hidden;"><span style="float:right; font-weight:600; font-size:0.85em; opacity:0.95;">5 min</span>✏️ Aufgabe 2.2 #3</div>
<div style="background:#EAF4EB; color:#1a1a1a; padding:12px 16px; margin:0;">
<p>Kodieren Sie das Fantasiewort <code>Akwirw ier</code> und speichern Sie die Token-IDs in <code>ids_fantasie</code>. Dekodieren Sie danach jede ID einzeln und speichern Sie die Stücke als Liste in <code>teile</code>.</p>
<p>Wer früher fertig ist: Setzen Sie die IDs mit <code>decode</code> wieder zum Originaltext zusammen. Geht dabei etwas verloren?</p>
<details style="margin-top:10px"><summary style="cursor:pointer; color:#2E7D32; font-weight:600; font-size:0.9em">Hinweis anzeigen</summary><div style="margin-top:8px; padding-top:8px; border-top:1px dashed #2E7D3255"><p><strong>Hinweis:</strong> <code>tokenizer.decode</code> erwartet eine Liste. Für eine einzelne ID lautet der Aufruf <code>tokenizer.decode([i])</code>.</p></div></details>

</div>
</div>

In [ ]:
# TODO: Ihre Lösung hier
ids_fantasie = ...
teile = ...

In [ ]:
# Prüfzelle — einfach ausführen
assert ids_fantasie == [33901, 86, 343, 86, 220, 959], "Kodieren Sie genau den Text Akwirw ier"
assert teile == ["Ak", "w", "ir", "w", " ", "ier"], "teile enthält die einzeln dekodierten Tokens"
assert "".join(teile) == "Akwirw ier"
print("Alle Prüfungen bestanden ✓")

<div style="border-radius:6px; overflow:hidden; margin:2px 0; margin-right:-20px;">
<div style="background:#2E7D32; color:#fff; padding:6px 16px; font-weight:bold; font-size:1.08em; margin:0; overflow:hidden;"><span style="float:right; font-weight:600; font-size:0.85em; opacity:0.95;">8 min</span>✏️ Aufgabe 2.2 #4</div>
<div style="background:#EAF4EB; color:#1a1a1a; padding:12px 16px; margin:0;">
<p>Untersuchen Sie den Tokenizer mit <code>zeige_tokens</code>.</p>
<ol>
<li>Welche Rolle spielen führende Leerzeichen und die Groß- und Kleinschreibung?</li>
<li>Wie werden deutsche Wörter zerlegt? Probieren Sie <code>Hochschule Worms</code>, <code>Donaudampfschifffahrtsgesellschaft</code> und ein Wort mit Umlaut.</li>
<li>Berechnen Sie für die beiden Sätze in der Zelle die Anzahl der Tokens pro Wort und speichern Sie die Werte in <code>tpw_en</code> und <code>tpw_de</code>.</li>
</ol>
<p>Wer früher fertig ist: Suchen Sie einen deutschen Satz, der möglichst viele Tokens pro Wort braucht.</p>
<details style="margin-top:10px"><summary style="cursor:pointer; color:#2E7D32; font-weight:600; font-size:0.9em">Hinweis anzeigen</summary><div style="margin-top:8px; padding-top:8px; border-top:1px dashed #2E7D3255"><p><strong>Hinweis:</strong> <code>satz.split()</code> liefert die Wörter eines Satzes als Liste.</p></div></details>

</div>
</div>

In [ ]:
en = "The students build a large language model from scratch."
de = "Die Studierenden bauen ein großes Sprachmodell von Grund auf."

# TODO: Ihre Lösung hier
tpw_en = ...
tpw_de = ...

In [ ]:
# Prüfzelle — einfach ausführen
assert abs(tpw_en - len(tokenizer.encode(en)) / 9) < 1e-9, "Tokens des englischen Satzes geteilt durch seine 9 Wörter"
assert abs(tpw_de - len(tokenizer.encode(de)) / 9) < 1e-9, "Tokens des deutschen Satzes geteilt durch seine 9 Wörter"
assert tpw_de > tpw_en, "Der deutsche Satz braucht mehr Tokens pro Wort"
print("Alle Prüfungen bestanden ✓")

<div style="border-radius:6px; overflow:hidden; margin:2px 0; margin-right:-20px;">
<div style="background:#2E7D32; color:#fff; padding:6px 16px; font-weight:bold; font-size:1.08em; margin:0; overflow:hidden;"><span style="float:right; font-weight:600; font-size:0.85em; opacity:0.95;">5 min</span>✏️ Aufgabe 2.2 #5</div>
<div style="background:#EAF4EB; color:#1a1a1a; padding:12px 16px; margin:0;">
<p>Führen Sie BPE von Hand auf den Wörtern <code>Heizung</code> und <code>Zeitung</code> aus. Jedes Wort kommt einmal vor. In welche Stücke sind die beiden Wörter nach drei Merges zerlegt? Tragen Sie Ihre Vorhersage als Listen ein und prüfen Sie sie mit <code>lerne_bpe</code>.</p>
<details style="margin-top:10px"><summary style="cursor:pointer; color:#2E7D32; font-weight:600; font-size:0.9em">Hinweis anzeigen</summary><div style="margin-top:8px; padding-top:8px; border-top:1px dashed #2E7D3255"><p><strong>Hinweis:</strong> Zählen Sie die Nachbarpaare über beide Wörter. Mehrere Paare kommen zweimal vor. Bei Gleichstand nimmt <code>lerne_bpe</code> das zuerst gezählte Paar.</p></div></details>

</div>
</div>

In [ ]:
# TODO: Ihre Lösung hier
heizung = [...]
zeitung = [...]

In [ ]:
# Prüfzelle — einfach ausführen
_merges, _ = lerne_bpe({"Heizung": 1, "Zeitung": 1}, 3)
assert heizung == zerlege("Heizung", _merges), "Drei Merges: Welche Paare kommen in beiden Wörtern vor?"
assert zeitung == zerlege("Zeitung", _merges), "Drei Merges: Welche Paare kommen in beiden Wörtern vor?"
print("Alle Prüfungen bestanden ✓")

<div style="border:2px solid #7E3FA0; background:#fff; color:#1a1a1a; padding:12px 16px; border-radius:6px; ">
<div style="overflow:hidden; margin-bottom:4px;"><span style="color:#7E3FA0; font-weight:bold; font-size:1.08em;">Lernfragen 2.2 #2</span></div>
<p style="padding-left:1.2em; text-indent:-1.2em;"><b style="color:#7E3FA0;">1.</b> Das Vokabular von GPT-2 hat 50.257 Einträge. Aus welchen drei Teilen setzt sich diese Zahl zusammen?</p>
<p style="padding-left:1.2em; text-indent:-1.2em;"><b style="color:#7E3FA0;">2.</b> Kontextlänge und Kosten eines Sprachmodells werden in Tokens gemessen. Was folgt aus Ihrem Ergebnis für deutsche Texte?</p>
<p style="padding-left:1.2em; text-indent:-1.2em;"><b style="color:#7E3FA0;">3.</b> <code>Hello</code>, <code> Hello</code> und <code>hello</code> sind drei verschiedene Tokens. Nennen Sie einen Vorteil und einen Nachteil dieser Entscheidung.</p>

</div>

## <a id="kap-trainingspaare-mit-dem-sliding-window"></a><span class="zwischentitel" style="display:block; color:#0E2841; border-bottom:3px solid #156082; padding-bottom:4px;  padding-top:26px;"><span style="float:right"><span class="modulchip" style="font-size:0.55em; font-weight:bold; letter-spacing:0.02em; border:1.5px solid #475569; color:#475569; padding:1px 8px; border-radius:10px; margin-left:12px; vertical-align:0.35em; white-space:nowrap;">Raschka 2.6</span></span>2.3&nbsp;&nbsp;Trainingspaare mit dem Sliding Window</span>

Ein GPT-Modell lernt, das nächste Token vorherzusagen. Das Ziel ist deshalb immer die Eingabe, um eine Position nach rechts verschoben. Zuerst wird der ganze Text in Token-IDs umgewandelt.

In [ ]:
enc_text = tokenizer.encode(raw_text)
print("Tokens im Text:", len(enc_text))

enc_sample = enc_text[100:]          # ein Ausschnitt ab Token 100
context_size = 4

x = enc_sample[:context_size]
y = enc_sample[1:context_size + 1]
print(f"x: {x}")
print(f"y:      {y}")

### <span class="zwischentitel" style="display:block; color:#0E2841; border-bottom:1px solid rgba(21,96,130,0.45); padding-bottom:2px;  padding-top:16px;">2.3.1&nbsp;&nbsp;Dataset und DataLoader</span>

<div style="border:2px solid #156082; background:#fff; color:#1a1a1a; padding:12px 16px; border-radius:6px; margin-right:-20px;">
<div style="overflow:hidden; margin-bottom:4px;"><span style="color:#156082; font-weight:bold; font-size:1.08em;">🔄 Wiederholung: Dataset und DataLoader</span></div>
<p>Ein <code>Dataset</code> liefert über <code>__getitem__</code> ein einzelnes Trainingsbeispiel und über <code>__len__</code> deren Anzahl. Der <code>DataLoader</code> stapelt mehrere Beispiele zu einem Batch. Eingeführt wurde das in ANN-1 bei der Trainingsschleife in PyTorch.</p>

</div>

`GPTDatasetV1` schiebt ein Fenster der Länge `max_length` über die Token-IDs. `stride` ist die Schrittweite, um die das Fenster weiterspringt.

In [ ]:
# Vorgabe: Dataset und DataLoader — einfach ausführen
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_length, "Der Text braucht mindestens max_length+1 Tokens"

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128,
                         shuffle=True, drop_last=True, num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                      drop_last=drop_last, num_workers=num_workers)


dataloader = create_dataloader_v1(raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)
data_iter = iter(dataloader)
print(next(data_iter))
print(next(data_iter))

<div style="border:2px solid #E97132; background:#fff; color:#1a1a1a; padding:12px 16px; border-radius:6px; margin-right:-20px;">
<div style="overflow:hidden; margin-bottom:4px;"><span style="color:#E97132; font-weight:bold; font-size:1.08em;">💭 Kurz überlegen</span></div>
<p>Der DataLoader liefert Batches mit <code>batch_size=8</code> und <code>max_length=4</code>. Welche Form hat der Eingabe-Tensor, und wie unterscheidet sich das Ziel von der Eingabe?</p>

</div>

In [ ]:
# Vorgabe: erster Batch — einfach ausführen
beispiel_loader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)
eingabe, ziel = next(iter(beispiel_loader))
print("Form der Eingabe:", tuple(eingabe.shape))
print("Eingabe:", eingabe[:2].tolist())
print("Ziel:   ", ziel[:2].tolist())

### <span class="zwischentitel" style="display:block; color:#0E2841; border-bottom:1px solid rgba(21,96,130,0.45); padding-bottom:2px;  padding-top:16px;">2.3.2&nbsp;&nbsp;Übungen</span>

<div style="border-radius:6px; overflow:hidden; margin:2px 0; margin-right:-20px;">
<div style="background:#2E7D32; color:#fff; padding:6px 16px; font-weight:bold; font-size:1.08em; margin:0; overflow:hidden;"><span style="float:right; font-weight:600; font-size:0.85em; opacity:0.95;">3 min</span>✏️ Aufgabe 2.3 #1</div>
<div style="background:#EAF4EB; color:#1a1a1a; padding:12px 16px; margin:0;">
<p>Ein Fenster der Länge 4 enthält vier Vorhersageaufgaben: aus einem Token das zweite, aus zwei Tokens das dritte und so weiter. Geben Sie alle vier als lesbaren Text aus, im Format <code>Kontext ----&gt; Ziel</code>.</p>
<details style="margin-top:10px"><summary style="cursor:pointer; color:#2E7D32; font-weight:600; font-size:0.9em">Hinweis anzeigen</summary><div style="margin-top:8px; padding-top:8px; border-top:1px dashed #2E7D3255"><p><strong>Hinweis:</strong> Die Schleife liefert <code>context</code> als Liste von IDs und <code>desired</code> als einzelne ID. <code>decode</code> erwartet in beiden Fällen eine Liste.</p></div></details>

</div>
</div>

In [ ]:
for i in range(1, context_size + 1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    # TODO: Ihre Lösung hier
    ...

<div style="border:2px solid #E97132; background:#fff; color:#1a1a1a; padding:12px 16px; border-radius:6px; margin-right:-20px;">
<div style="overflow:hidden; margin-bottom:4px;"><span style="color:#E97132; font-weight:bold; font-size:1.08em;">💭 Kurz überlegen</span></div>
<p>Die ersten Token-IDs des Textes lauten <code>40, 367, 2885, 1464, 1807, 3619, 402, 271, 10899, 2138</code>. Schreiben Sie auf, wie der Eingabe-Tensor des ersten Batches bei <code>batch_size=4</code> aussieht, einmal für <code>max_length=2, stride=2</code> und einmal für <code>max_length=8, stride=2</code>.</p>

</div>

<div style="border-radius:6px; overflow:hidden; margin:2px 0; margin-right:-20px;">
<div style="background:#2E7D32; color:#fff; padding:6px 16px; font-weight:bold; font-size:1.08em; margin:0; overflow:hidden;"><span style="float:right; font-weight:600; font-size:0.85em; opacity:0.95;">7 min</span>✏️ Aufgabe 2.3 #2</div>
<div style="background:#EAF4EB; color:#1a1a1a; padding:12px 16px; margin:0;">
<p>Prüfen Sie Ihre Vorhersage. Erzeugen Sie für beide Einstellungen einen DataLoader mit <code>shuffle=False</code> und speichern Sie den Eingabe-Tensor des ersten Batches in <code>x_kurz</code> (für <code>max_length=2</code>) und <code>x_lang</code> (für <code>max_length=8</code>).</p>
<details style="margin-top:10px"><summary style="cursor:pointer; color:#2E7D32; font-weight:600; font-size:0.9em">Hinweis anzeigen</summary><div style="margin-top:8px; padding-top:8px; border-top:1px dashed #2E7D3255"><p><strong>Hinweis:</strong> <code>next(iter(dataloader))</code> liefert das Paar aus Eingabe und Ziel des ersten Batches.</p></div></details>

</div>
</div>

In [ ]:
# TODO: Ihre Lösung hier
x_kurz = ...
x_lang = ...

In [ ]:
# Prüfzelle — einfach ausführen
assert x_kurz.tolist() == [[40, 367], [2885, 1464], [1807, 3619], [402, 271]]
assert tuple(x_lang.shape) == (4, 8), "vier Zeilen mit je acht Tokens"
assert x_lang[1].tolist() == [2885, 1464, 1807, 3619, 402, 271, 10899, 2138], "jede Zeile beginnt zwei Tokens später"
print("Alle Prüfungen bestanden ✓")

<div style="border-radius:6px; overflow:hidden; margin:2px 0; margin-right:-20px;">
<div style="background:#2E7D32; color:#fff; padding:6px 16px; font-weight:bold; font-size:1.08em; margin:0; overflow:hidden;"><span style="float:right; font-weight:600; font-size:0.85em; opacity:0.95;">5 min</span>✏️ Aufgabe 2.3 #3</div>
<div style="background:#EAF4EB; color:#1a1a1a; padding:12px 16px; margin:0;">
<p>Wie viele Trainingsbeispiele enthält das Dataset bei <code>max_length=4</code>, einmal für <code>stride=1</code> und einmal für <code>stride=4</code>? Leiten Sie beide Zahlen aus der Anzahl der Tokens im Text her und speichern Sie sie in <code>n_stride1</code> und <code>n_stride4</code>. Verwenden Sie dafür nicht <code>len(dataset)</code>, das übernimmt die Prüfzelle.</p>
<details style="margin-top:10px"><summary style="cursor:pointer; color:#2E7D32; font-weight:600; font-size:0.9em">Hinweis anzeigen</summary><div style="margin-top:8px; padding-top:8px; border-top:1px dashed #2E7D3255"><p><strong>Hinweis:</strong> Das Fenster startet bei 0, <code>stride</code>, <code>2 * stride</code> und so weiter. Nach dem Fenster muss noch ein Token als Ziel übrig sein.</p></div></details>

</div>
</div>

In [ ]:
# TODO: Ihre Lösung hier
n_stride1 = ...
n_stride4 = ...

In [ ]:
# Prüfzelle — einfach ausführen
_tok = tiktoken.get_encoding("gpt2")
assert n_stride1 == len(GPTDatasetV1(raw_text, _tok, max_length=4, stride=1))
assert n_stride4 == len(GPTDatasetV1(raw_text, _tok, max_length=4, stride=4))
print("Alle Prüfungen bestanden ✓")

<div style="border:2px solid #7E3FA0; background:#fff; color:#1a1a1a; padding:12px 16px; border-radius:6px; ">
<div style="overflow:hidden; margin-bottom:4px;"><span style="color:#7E3FA0; font-weight:bold; font-size:1.08em;">Lernfragen 2.3 #1</span></div>
<p style="padding-left:1.2em; text-indent:-1.2em;"><b style="color:#7E3FA0;">1.</b> Was bewirkt ein <code>stride</code>, der kleiner ist als <code>max_length</code>? Nennen Sie einen Vorteil und einen Nachteil.</p>
<p style="padding-left:1.2em; text-indent:-1.2em;"><b style="color:#7E3FA0;">2.</b> Wozu dienen <code>shuffle=True</code> und <code>drop_last=True</code> beim Training?</p>
<p style="padding-left:1.2em; text-indent:-1.2em;"><b style="color:#7E3FA0;">3.</b> Für das Training wurden keine Labels von Hand vergeben. Woher kommen sie?</p>

</div>

## <a id="kap-embeddings"></a><span class="zwischentitel" style="display:block; color:#0E2841; border-bottom:3px solid #156082; padding-bottom:4px;  padding-top:26px;"><span style="float:right"><span class="modulchip" style="font-size:0.55em; font-weight:bold; letter-spacing:0.02em; border:1.5px solid #475569; color:#475569; padding:1px 8px; border-radius:10px; margin-left:12px; vertical-align:0.35em; white-space:nowrap;">Raschka 2.7 und 2.8</span></span>2.4&nbsp;&nbsp;Embeddings</span>

Token-IDs sind nur Nummern. Eine ID von 4000 ist nicht größer oder wichtiger als eine ID von 40. Das Modell braucht für jedes Token einen Vektor, dessen Werte es beim Training selbst lernt. Ein kleines Beispiel mit 6 Tokens im Vokabular und 3 Dimensionen:

In [ ]:
input_ids = torch.tensor([2, 3, 5, 1])

vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

print(embedding_layer.weight)
print(embedding_layer(torch.tensor([3])))

<div style="border:2px solid #E97132; background:#fff; color:#1a1a1a; padding:12px 16px; border-radius:6px; margin-right:-20px;">
<div style="overflow:hidden; margin-bottom:4px;"><span style="color:#E97132; font-weight:bold; font-size:1.08em;">💭 Kurz überlegen</span></div>
<p>GPT-2 small hat ein Vokabular von 50.257 Tokens und Vektoren mit 768 Dimensionen. Wie viele Parameter hat die Embedding-Tabelle?</p>

</div>

In [ ]:
# Vorgabe: Embedding-Tabelle in GPT-2-Größe — einfach ausführen
gpt2_tabelle = torch.nn.Embedding(50257, 768)
print("Parameter:", gpt2_tabelle.weight.numel())
print("Anteil an GPT-2 small:", f"{gpt2_tabelle.weight.numel() / 124e6:.0%}")
print("Form für eine einzelne ID:", tuple(gpt2_tabelle(torch.tensor([617])).shape))

### <span class="zwischentitel" style="display:block; color:#0E2841; border-bottom:1px solid rgba(21,96,130,0.45); padding-bottom:2px;  padding-top:16px;">2.4.1&nbsp;&nbsp;Token- und Positions-Embeddings in GPT-Größe</span>

Jetzt mit dem echten Vokabular: 50.257 Tokens, 256 Dimensionen und Batches aus 8 Sequenzen mit je 4 Tokens.

In [ ]:
vocab_size = 50257
output_dim = 256
max_length = 4

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=max_length,
                                  stride=max_length, shuffle=False)
inputs, targets = next(iter(dataloader))
print("inputs:", inputs.shape)

### <span class="zwischentitel" style="display:block; color:#0E2841; border-bottom:1px solid rgba(21,96,130,0.45); padding-bottom:2px;  padding-top:16px;">2.4.2&nbsp;&nbsp;Warum Positionen?</span>

Die beiden Sätze in der nächsten Zelle enthalten dieselben Wörter in anderer Reihenfolge. Das Token ` dog` steht in Satz 1 an Position `p1` und in Satz 2 an Position `p2`.

In [ ]:
torch.manual_seed(123)
emb = torch.nn.Embedding(50257, 16)
pos = torch.nn.Embedding(16, 16)

ids1 = torch.tensor(tokenizer.encode("the dog bites the man"))
ids2 = torch.tensor(tokenizer.encode("the man bites the dog"))

dog_id = tokenizer.encode(" dog")[0]
p1 = ids1.tolist().index(dog_id)
p2 = ids2.tolist().index(dog_id)
print("Positionen von ' dog':", p1, p2)

### <span class="zwischentitel" style="display:block; color:#0E2841; border-bottom:1px solid rgba(21,96,130,0.45); padding-bottom:2px;  padding-top:16px;">2.4.3&nbsp;&nbsp;Übungen</span>

<div style="border-radius:6px; overflow:hidden; margin:2px 0; margin-right:-20px;">
<div style="background:#2E7D32; color:#fff; padding:6px 16px; font-weight:bold; font-size:1.08em; margin:0; overflow:hidden;"><span style="float:right; font-weight:600; font-size:0.85em; opacity:0.95;">5 min</span>✏️ Aufgabe 2.4 #1</div>
<div style="background:#EAF4EB; color:#1a1a1a; padding:12px 16px; margin:0;">
<p>Zeigen Sie zwei Dinge mit je einer Zeile Code.</p>
<ol>
<li>Das Embedding von Token 3 ist genau die Zeile mit Index 3 der Gewichtsmatrix. Speichern Sie das Ergebnis des Vergleichs in <code>ist_zeile</code>.</li>
<li>Ein Lookup liefert dasselbe wie One-Hot-Vektor mal Gewichtsmatrix. Berechnen Sie das Produkt für <code>input_ids</code> und speichern Sie es in <code>per_matmul</code>.</li>
</ol>
<details style="margin-top:10px"><summary style="cursor:pointer; color:#2E7D32; font-weight:600; font-size:0.9em">Hinweis anzeigen</summary><div style="margin-top:8px; padding-top:8px; border-top:1px dashed #2E7D3255"><p><strong>Hinweis:</strong> <code>torch.equal(a, b)</code> vergleicht zwei Tensoren. <code>torch.nn.functional.one_hot(ids, num_classes=6)</code> liefert ganze Zahlen, für die Multiplikation mit <code>@</code> wandelt <code>.float()</code> sie um.</p></div></details>

</div>
</div>

In [ ]:
# TODO: Ihre Lösung hier
ist_zeile = ...
per_matmul = ...

In [ ]:
# Prüfzelle — einfach ausführen
assert ist_zeile is True
assert torch.allclose(per_matmul, embedding_layer(input_ids)), "Lookup und One-Hot-Multiplikation liefern dasselbe"
print("Alle Prüfungen bestanden ✓")

<div style="border:2px solid #E97132; background:#fff; color:#1a1a1a; padding:12px 16px; border-radius:6px; margin-right:-20px;">
<div style="overflow:hidden; margin-bottom:4px;"><span style="color:#E97132; font-weight:bold; font-size:1.08em;">💭 Kurz überlegen</span></div>
<p>Welche Form haben die Token-Embeddings für diesen Batch, welche Form haben die Positions-Embeddings für die Positionen 0 bis 3, und welche Form hat ihre Summe?</p>

</div>

<div style="border-radius:6px; overflow:hidden; margin:2px 0; margin-right:-20px;">
<div style="background:#2E7D32; color:#fff; padding:6px 16px; font-weight:bold; font-size:1.08em; margin:0; overflow:hidden;"><span style="float:right; font-weight:600; font-size:0.85em; opacity:0.95;">5 min</span>✏️ Aufgabe 2.4 #2</div>
<div style="background:#EAF4EB; color:#1a1a1a; padding:12px 16px; margin:0;">
<p>Legen Sie einen zweiten Embedding-Layer <code>pos_embedding_layer</code> für die Positionen an. Berechnen Sie <code>pos_embeddings</code> für die Positionen 0 bis 3 und bilden Sie <code>input_embeddings</code> als Summe aus Token- und Positions-Embeddings.</p>
<details style="margin-top:10px"><summary style="cursor:pointer; color:#2E7D32; font-weight:600; font-size:0.9em">Hinweis anzeigen</summary><div style="margin-top:8px; padding-top:8px; border-top:1px dashed #2E7D3255"><p><strong>Hinweis:</strong> <code>torch.arange(max_length)</code> liefert die Positionen 0 bis 3 als Tensor.</p></div></details>

</div>
</div>

In [ ]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

# TODO: Ihre Lösung hier
pos_embedding_layer = ...
pos_embeddings = ...
input_embeddings = ...

In [ ]:
# Prüfzelle — einfach ausführen
assert tuple(pos_embeddings.shape) == (4, 256), "ein Vektor je Position"
assert tuple(input_embeddings.shape) == (8, 4, 256), "Batch, Sequenzlänge, Dimension"
assert torch.allclose(input_embeddings[0, 2], token_embeddings[0, 2] + pos_embeddings[2])
print("Alle Prüfungen bestanden ✓")

<div style="border-radius:6px; overflow:hidden; margin:2px 0; margin-right:-20px;">
<div style="background:#2E7D32; color:#fff; padding:6px 16px; font-weight:bold; font-size:1.08em; margin:0; overflow:hidden;"><span style="float:right; font-weight:600; font-size:0.85em; opacity:0.95;">5 min</span>✏️ Aufgabe 2.4 #3</div>
<div style="background:#EAF4EB; color:#1a1a1a; padding:12px 16px; margin:0;">
<p>GPT-2 small verwendet Embeddings mit 768 Dimensionen und eine Kontextlänge von 1.024 Tokens. Berechnen Sie die Anzahl der Parameter der Token-Embeddings (<code>tok_params</code>) und der Positions-Embeddings (<code>pos_params</code>). Das ganze Modell hat rund 124 Millionen Parameter. Welchen Anteil machen die Embeddings aus (<code>anteil</code>, als Zahl zwischen 0 und 1)?</p>

</div>
</div>

In [ ]:
# TODO: Ihre Lösung hier
tok_params = ...
pos_params = ...
anteil = ...

In [ ]:
# Prüfzelle — einfach ausführen
assert tok_params == 38_597_376
assert pos_params == 786_432
assert 0.30 < anteil < 0.33, "knapp ein Drittel"
print("Alle Prüfungen bestanden ✓")

<div style="border:2px solid #E97132; background:#fff; color:#1a1a1a; padding:12px 16px; border-radius:6px; margin-right:-20px;">
<div style="overflow:hidden; margin-bottom:4px;"><span style="color:#E97132; font-weight:bold; font-size:1.08em;">💭 Kurz überlegen</span></div>
<p>Ist der Vektor für <code> dog</code> in beiden Sätzen derselbe? Beantworten Sie die Frage einmal für reine Token-Embeddings und einmal für die Summe aus Token- und Positions-Embeddings.</p>

</div>

<div style="border-radius:6px; overflow:hidden; margin:2px 0; margin-right:-20px;">
<div style="background:#2E7D32; color:#fff; padding:6px 16px; font-weight:bold; font-size:1.08em; margin:0; overflow:hidden;"><span style="float:right; font-weight:600; font-size:0.85em; opacity:0.95;">7 min</span>✏️ Aufgabe 2.4 #4</div>
<div style="background:#EAF4EB; color:#1a1a1a; padding:12px 16px; margin:0;">
<p>Vergleichen Sie den Vektor für <code> dog</code> in beiden Sätzen. Speichern Sie in <code>ohne_pos</code>, ob die Vektoren ohne Positions-Embeddings gleich sind, und in <code>mit_pos</code>, ob sie mit Positions-Embeddings gleich sind.</p>
<details style="margin-top:10px"><summary style="cursor:pointer; color:#2E7D32; font-weight:600; font-size:0.9em">Hinweis anzeigen</summary><div style="margin-top:8px; padding-top:8px; border-top:1px dashed #2E7D3255"><p><strong>Hinweis:</strong> <code>emb(ids1)[p1]</code> ist der Vektor für <code> dog</code> in Satz 1. Die Positions-Embeddings für einen Satz liefert <code>pos(torch.arange(len(ids1)))</code>.</p></div></details>

</div>
</div>

In [ ]:
# TODO: Ihre Lösung hier
ohne_pos = ...
mit_pos = ...

In [ ]:
# Prüfzelle — einfach ausführen
assert ohne_pos is True, "dieselbe Token-ID liefert denselben Vektor"
assert mit_pos is False, "an verschiedenen Positionen kommen verschiedene Positionsvektoren dazu"
print("Alle Prüfungen bestanden ✓")

<div style="border:2px solid #7E3FA0; background:#fff; color:#1a1a1a; padding:12px 16px; border-radius:6px; ">
<div style="overflow:hidden; margin-bottom:4px;"><span style="color:#7E3FA0; font-weight:bold; font-size:1.08em;">Lernfragen 2.4 #1</span></div>
<p style="padding-left:1.2em; text-indent:-1.2em;"><b style="color:#7E3FA0;">1.</b> Warum lässt sich ein Tensor der Form <code>[8, 4, 256]</code> mit einem Tensor der Form <code>[4, 256]</code> addieren?</p>
<p style="padding-left:1.2em; text-indent:-1.2em;"><b style="color:#7E3FA0;">2.</b> Lookup und One-Hot-Multiplikation liefern dasselbe Ergebnis. Warum verwendet man den Lookup?</p>
<p style="padding-left:1.2em; text-indent:-1.2em;"><b style="color:#7E3FA0;">3.</b> Ein RNN aus ANN-1 braucht keine Positions-Embeddings. Warum nicht?</p>

</div>

<div style="border-radius:6px; overflow:hidden; margin:2px 0; margin-right:-20px;">
<div style="background:#107C72; color:#fff; padding:6px 16px; font-weight:bold; font-size:1.08em; margin:0; overflow:hidden;"><span style="float:right"><span style="border:1px solid rgba(255,255,255,0.85); color:#fff; font-size:0.72em; font-weight:bold; padding:1px 9px; border-radius:10px; margin-left:8px; vertical-align:1px;">NICHT PRÜFUNGSRELEVANT</span></span>Vertiefung: Absolute und relative Positionen</div>
<div style="background:#E7F4F2; color:#1a1a1a; padding:12px 16px; margin:0;">
<p>GPT-2 lernt für jede Position von 0 bis 1.023 einen eigenen Vektor. Das Modell kann deshalb keine längeren Eingaben verarbeiten, und es muss für jede Position neu lernen, was Nachbarschaft bedeutet.</p>
<p>Neuere Modelle wie Llama oder Qwen kodieren stattdessen den <strong>Abstand</strong> zwischen zwei Tokens, meist mit Rotary Position Embeddings (RoPE). Dort wird die Position nicht zur Eingabe addiert, sondern in jedem Attention-Schritt eingerechnet. RoPE ist die Grundlage eines der Seminarprojekte.</p>

</div>
</div>

<div style="border-radius:6px; overflow:hidden; margin:2px 0; margin-right:-20px;">
<div style="background:#107C72; color:#fff; padding:6px 16px; font-weight:bold; font-size:1.08em; margin:0; overflow:hidden;"><span style="float:right"><span style="border:1px solid rgba(255,255,255,0.85); color:#fff; font-size:0.72em; font-weight:bold; padding:1px 9px; border-radius:10px; margin-left:8px; vertical-align:1px;">NICHT PRÜFUNGSRELEVANT</span></span>Vertiefung: Ein Chat-Template von Hand</div>
<div style="background:#E7F4F2; color:#1a1a1a; padding:12px 16px; margin:0;">
<p>Ein Chat-Modell sieht keine Nachrichtenliste, sondern eine einzige Token-Folge. Das Chat-Template setzt die Nachrichten mit Sondertokens zu einer Zeichenkette zusammen. Die nächste Zelle baut eine solche Zeichenkette im ChatML-Format.</p>
<p>GPT-2 ist kein Chat-Modell und kennt diese Sondertokens nicht. Sein Tokenizer zerlegt sie deshalb in gewöhnliche Stücke. Ein Chat-Modell hat für jedes dieser Sondertokens genau eine ID.</p>

</div>
</div>

In [ ]:
nachrichten = [{"role": "user", "content": "Was ist ein Token?"}]

prompt = ""
for nachricht in nachrichten:
    prompt += f"<|im_start|>{nachricht['role']}\n{nachricht['content']}<|im_end|>\n"
prompt += "<|im_start|>assistant\n"        # hier setzt das Modell fort

print(prompt)
_ = zeige_tokens("<|im_start|>")

<div style="border-radius:6px; overflow:hidden; margin:2px 0; margin-right:-20px;">
<div style="background:#E8B029; color:#fff; padding:6px 16px; font-weight:bold; font-size:1.08em; margin:0; overflow:hidden;">➡️ Nächste Schritte</div>
<div style="background:#FDF6E3; color:#1a1a1a; padding:12px 16px; margin:0;">
<ul>
<li>Bearbeiten Sie die Übungen, die im Termin offen geblieben sind. Zur Schlusskontrolle starten Sie den Kernel neu und führen alle Zellen aus: Alle Prüfzellen sollten mit Ihren eigenen Lösungen durchlaufen.</li>
<li>Lesen Sie Raschka, Kapitel 3 bis einschließlich Abschnitt 3.3.</li>
<li>Tragen Sie Ihre drei Themenfavoriten für das Seminarprojekt und Ihr Team im Moodle ein.</li>
</ul>
<p><strong>→ Ausblick:</strong> Das Ergebnis von heute ist ein Tensor der Form <code>[B, T, d]</code>: Batch, Sequenzlänge, Dimension. In der nächsten Lerneinheit ist er die Eingabe für den Attention-Mechanismus.</p>

</div>
</div>

<div style="border-radius:6px; overflow:hidden; margin:2px 0; margin-right:-20px;">
<div style="background:#E8B029; color:#fff; padding:6px 16px; font-weight:bold; font-size:1.08em; margin:0; overflow:hidden;">Quellen und Referenzen</div>
<div style="background:#FDF6E3; color:#1a1a1a; padding:12px 16px; margin:0;">
<ul>
<li>Sebastian Raschka: *Build a Large Language Model (From Scratch)*, Manning 2024, Kapitel 2. Code: <a href="https://github.com/rasbt/LLMs-from-scratch/tree/main/ch02">rasbt/LLMs-from-scratch</a>, Lizenz Apache 2.0. Der Code der Kapitel 2.2 bis 2.4 ist daraus übernommen und angepasst.</li>
<li>Rico Sennrich, Barry Haddow, Alexandra Birch: *Neural Machine Translation of Rare Words with Subword Units*, ACL 2016. Der Algorithmus in Kapitel 2.2 folgt Algorithmus 1.</li>
<li>Claude E. Shannon: *Prediction and Entropy of Printed English*, Bell System Technical Journal 30(1), 1951. Die Idee zu Kapitel 2.1.</li>
<li>Zum Vertiefen: <a href="https://github.com/rasbt/LLMs-from-scratch/tree/main/ch02/05_bpe-from-scratch">BPE from scratch</a> und <a href="https://github.com/rasbt/LLMs-from-scratch/tree/main/ch02/04_bonus_dataloader-intuition">Dataloader-Intuition</a> aus Raschkas Repository.</li>
<li>Beispieltexte: Edith Wharton, *The Verdict* (1908), und Texte von Johann Wolfgang von Goethe, beide gemeinfrei.</li>
</ul>

</div>
</div>